<a href="https://colab.research.google.com/github/A-Kuo/Language-Model-Hallucination-Detection-via-Entropy-Divergence/blob/main/colab/gpu_benchmark.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# AED Full GPU Benchmark
## Attention Entropy Divergence — Hallucination Detection

**Runtime:** ~15 min on T4 GPU (free Colab tier)  
**Output:** `results/benchmark_results.json` — auto-commits to repo with GitHub token

### What this does
1. Loads **Pythia-160m** (the model used in the paper)
2. Loads **HaluEval QA** (the benchmark used in the paper)
3. Extracts per-head Shannon entropy and cross-layer KL divergence
4. Trains BiLSTM and logistic regression classifiers
5. Reports AUROC, FPR@90%TPR, F1, and latency
6. **Auto-commits results back to repo** (with your GH_TOKEN)

### Before you run
**Runtime → Change runtime type → T4 GPU**

### Setup GH_TOKEN (for auto-commit)
1. Go to https://github.com/settings/tokens → Generate new token (classic)
2. Check scopes: `repo` (full control of private repositories)
3. Copy token → Secrets tab (🔑) on left sidebar → Add `GH_TOKEN`
4. Token allows the notebook to push results back to the repo

In [ ]:
# ── Load GitHub Token from Secrets ─────────────────────────────────────────
# Set GH_TOKEN in the Secrets tab (🔑) for auto-commit capability
from google.colab import userdata
import os

try:
    GH_TOKEN = userdata.get('GH_TOKEN')
    os.environ['GH_TOKEN'] = GH_TOKEN
    print('✓ GH_TOKEN loaded from Colab Secrets')
    print('  Auto-commit to repo: ENABLED')
except Exception as e:
    GH_TOKEN = None
    print('⚠️  GH_TOKEN not found in Secrets')
    print('  Auto-commit: DISABLED (manual download only)')
    print('  To enable: Secrets tab → Add GH_TOKEN with repo scope')

In [ ]:
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode != 0:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU')
else:
    print('✓ GPU detected:', result.stdout.strip())

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
%pip install -q transformers datasets accelerate
print('✓ Dependencies installed')

In [ ]:
import os, sys

REPO = 'Language-Model-Hallucination-Detection-via-Entropy-Divergence'
if not os.path.exists(REPO):
    !git clone https://github.com/A-Kuo/{REPO}.git
else:
    !git -C {REPO} pull

os.chdir(REPO)
sys.path.insert(0, 'v1')
print('✓ Repo ready:', os.getcwd())

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

!python v1/run_experiment.py \
    --mode full \
    --model EleutherAI/pythia-160m \
    --max-samples 500 \
    --output results/benchmark_results.json

In [ ]:
import json

with open('results/benchmark_results.json') as f:
    r = json.load(f)

print('='*60)
print('  FINAL RESULTS — Paper Placeholders')
print('='*60)
print(f'\RESULT{{{r["aed_auroc"]:.3f}}}  ← AED BiLSTM AUROC')
print(f'\RESULT{{{r["logreg_auroc"]:.3f}}}  ← LogReg Baseline AUROC')
print(f'{r["aed_latency_ms"]:.1f} ms    ← Latency per sample')
print('='*60)

In [ ]:
# Visualization cell (generates paper figures)
import numpy as np
import matplotlib.pyplot as plt
from run_experiment import extract_features, load_halueval_qa
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained('EleutherAI/pythia-160m')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained('EleutherAI/pythia-160m', output_attentions=True)
model.eval().cuda()

pairs = load_halueval_qa(max_samples=40)
num_layers = model.config.num_hidden_layers
correct_entropy = np.zeros(num_layers)
halluc_entropy = np.zeros(num_layers)
correct_count = halluc_count = 0

for ctx, cont, label in pairs:
    feats = extract_features(model, tokenizer, ctx, cont, 'cuda')
    if label == 0:
        correct_entropy += feats[:num_layers]
        correct_count += 1
    else:
        halluc_entropy += feats[:num_layers]
        halluc_count += 1

fig, ax = plt.subplots(figsize=(10, 6))
layers = range(1, num_layers + 1)
ax.plot(layers, correct_entropy/correct_count, 'b-o', label='Non-hallucinated')
ax.plot(layers, halluc_entropy/halluc_count, 'r-o', label='Hallucinated')
ax.set_xlabel('Layer')
ax.set_ylabel('Mean Attention Entropy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.savefig('results/figure_layer_entropy.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: results/figure_layer_entropy.png')

In [ ]:
# ── Auto-commit results back to GitHub ───────────────────────────────────
# Only runs if GH_TOKEN was provided in Secrets

import subprocess
import os

GH_TOKEN = os.environ.get('GH_TOKEN')

if not GH_TOKEN:
    print('⚠️  No GH_TOKEN — skipping auto-commit')
    print('   Results saved locally. Download manually:')
    from google.colab import files
    for f in ['results/benchmark_results.json', 'results/figure_layer_entropy.png']:
        if os.path.exists(f):
            files.download(f)
else:
    # Configure git with token
    remote_url = f'https://{GH_TOKEN}@github.com/A-Kuo/Language-Model-Hallucination-Detection-via-Entropy-Divergence.git'
    !git remote set-url origin {remote_url}
    !git config user.email "colab@gpu-benchmark.ai"
    !git config user.name "GPU Benchmark Bot"
    
    # Stage and commit results
    !git add results/
    commit_msg = f'results: GPU benchmark Pythia-160m + HaluEval (AUROC={r["aed_auroc"]:.3f})'
    !git commit -m "{commit_msg}"
    !git push origin main
    
    print('✓ Results committed to repo!')
    print('  https://github.com/A-Kuo/Language-Model-Hallucination-Detection-via-Entropy-Divergence/tree/main/results')